In [1]:
from pprint import pprint
from DbConnector import DbConnector
from collections import defaultdict


class ActorPairAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def actor_pairs_costarring(self):
        """
        Find actor pairs that co-starred in >= 3 movies together.
        Report: actor names, co-appearance count, average vote_average.
        Sort by co-appearance count (descending).
        """
        print("\n=== Query 2: Actor Pairs Co-starring ===\n")
        
        # Step 1: Fetch all movies with their cast and vote_average
        pipeline = [
            {
                "$lookup": {
                    "from": "movies",
                    "localField": "_id",
                    "foreignField": "_id",
                    "as": "movie_info"
                }
            },
            {"$unwind": "$movie_info"},
            {
                "$project": {
                    "cast": 1,
                    "vote_average": "$movie_info.vote_average",
                    "title": "$movie_info.title"
                }
            }
        ]
        
        movies = list(self.db.credits.aggregate(pipeline))
        
        # Step 2: Build actor pairs with Python
        # pair_data[frozenset({actor1, actor2})] = {movies: [...], vote_averages: [...]}
        pair_data = defaultdict(lambda: {"movies": [], "vote_averages": []})
        
        for movie in movies:
            cast = movie.get("cast", [])
            vote_avg = movie.get("vote_average")
            
            # Get unique actor names (filter out None/empty)
            actors = []
            seen_ids = set()
            for actor in cast:
                actor_id = actor.get("id")
                actor_name = actor.get("name")
                if actor_id and actor_name and actor_id not in seen_ids:
                    actors.append((actor_id, actor_name))
                    seen_ids.add(actor_id)
            
            # Generate all pairs of actors in this movie
            for i in range(len(actors)):
                for j in range(i + 1, len(actors)):
                    actor1_id, actor1_name = actors[i]
                    actor2_id, actor2_name = actors[j]
                    
                    # Use frozenset to ensure (A,B) and (B,A) are treated as same pair
                    pair_key = frozenset({actor1_id})
                    pair_key = frozenset({(actor1_id, actor1_name), (actor2_id, actor2_name)})
                    
                    pair_data[pair_key]["movies"].append(movie["_id"])
                    if vote_avg is not None:
                        pair_data[pair_key]["vote_averages"].append(vote_avg)
        
        # Step 3: Filter pairs with >= 3 co-appearances
        results = []
        for pair_key, data in pair_data.items():
            co_appearance_count = len(data["movies"])
            
            if co_appearance_count >= 3:
                # Extract actor names from frozenset
                actors_list = list(pair_key)
                actor1_name = actors_list[0][1]
                actor2_name = actors_list[1][1]
                
                # Calculate average vote_average
                avg_vote = (sum(data["vote_averages"]) / len(data["vote_averages"]) 
                           if data["vote_averages"] else None)
                
                results.append({
                    "actor1": actor1_name,
                    "actor2": actor2_name,
                    "co_appearances": co_appearance_count,
                    "avg_vote_average": round(avg_vote, 2) if avg_vote else None
                })
        
        # Step 4: Sort by co-appearance count (descending)
        results.sort(key=lambda x: x["co_appearances"], reverse=True)
        
        # Step 5: Print results
        print(f"{'Rank':<5} {'Actor 1':<30} {'Actor 2':<30} {'Co-appearances':<15} {'Avg Vote':<10}")
        print("-" * 95)
        
        for i, pair in enumerate(results[:20], 1):  # Show top 20
            print(f"{i:<5} {pair['actor1']:<30} {pair['actor2']:<30} "
                  f"{pair['co_appearances']:<15} {pair['avg_vote_average']:<10}")
        
        print(f"\nTotal pairs found: {len(results)}")
        
        return results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = ActorPairAnalysis()
        results = program.actor_pairs_costarring()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 2: Actor Pairs Co-starring ===

Rank  Actor 1                        Actor 2                        Co-appearances  Avg Vote  
-----------------------------------------------------------------------------------------------
1     Leo Gorcey                     Huntz Hall                     35              6.35      
2     Charlie Chaplin                Edna Purviance                 33              6.46      
3     Stan Laurel                    Oliver Hardy                   31              6.35      
4     Rob Paulsen                    Jeff Bennett                   27              6.23      
5     Bud Abbott                     Lou Costello                   27              6.47      
6     Grey Griffin                   Frank Welker                   25              6.71      
7     Barbara Hale                   Raymond Burr                   25              5.93      
8     John Wayne                     Paul Fix            

In [2]:
from pprint import pprint
from DbConnector import DbConnector


class GenreBreadthAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def top_actors_by_genre_breadth(self):
        """
        Find top 10 actors (with >= 10 credited movies) with widest genre breadth.
        Report: actor name, number of distinct genres, up to 5 example genres.
        """
        print("\n=== Query 3: Actors with Widest Genre Breadth ===\n")
        
        # MongoDB aggregation pipeline
        pipeline = [
            # Step 1: Unwind cast array
            {"$unwind": "$cast"},
            
            # Step 2: Lookup movies to get genres
            {
                "$lookup": {
                    "from": "movies",
                    "localField": "_id",
                    "foreignField": "_id",
                    "as": "movie_info"
                }
            },
            {"$unwind": "$movie_info"},
            
            # Step 3: Unwind genres array
            {"$unwind": "$movie_info.genres"},
            
            # Step 4: Group by actor
            {
                "$group": {
                    "_id": {
                        "actor_id": "$cast.id",
                        "actor_name": "$cast.name"
                    },
                    "genres": {"$addToSet": "$movie_info.genres.name"},
                    "movie_count": {"$addToSet": "$_id"}  # Count distinct movies
                }
            },
            
            # Step 5: Project and calculate counts
            {
                "$project": {
                    "actor_id": "$_id.actor_id",
                    "actor_name": "$_id.actor_name",
                    "genres": 1,
                    "genre_count": {"$size": "$genres"},
                    "movie_count": {"$size": "$movie_count"},
                    "_id": 0
                }
            },
            
            # Step 6: Filter actors with >= 10 movies
            {"$match": {"movie_count": {"$gte": 10}}},
            
            # Step 7: Sort by genre count descending
            {"$sort": {"genre_count": -1}},
            
            # Step 8: Limit to top 10
            {"$limit": 10}
        ]
        
        results = list(self.db.credits.aggregate(pipeline))
        
        # Print results
        print(f"{'Rank':<5} {'Actor':<35} {'Movies':<8} {'Genres':<8} {'Example Genres':<50}")
        print("-" * 110)
        
        for i, actor in enumerate(results, 1):
            # Get up to 5 example genres
            example_genres = ', '.join(actor['genres'][:5])
            
            print(f"{i:<5} {actor['actor_name']:<35} {actor['movie_count']:<8} "
                  f"{actor['genre_count']:<8} {example_genres:<50}")
        
        return results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = GenreBreadthAnalysis()
        results = program.top_actors_by_genre_breadth()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 3: Actors with Widest Genre Breadth ===

Rank  Actor                               Movies   Genres   Example Genres                                    
--------------------------------------------------------------------------------------------------------------
1     Martin Sheen                        76       20       Fantasy, Music, Comedy, TV Movie, Adventure       
2     Christopher Lee                     145      20       Thriller, Family, Music, Drama, Animation         
3     Charlton Heston                     68       20       Thriller, Western, Animation, Music, War          
4     Ned Beatty                          70       20       Comedy, TV Movie, Crime, Romance, Horror          
5     Stacy Keach                         59       20       Fantasy, History, Science Fiction, Horror, Crime  
6     Eddie Albert                        47       20       Music, Action, War, Foreign, Animation            
7     Michael Ga

In [3]:
from pprint import pprint
from DbConnector import DbConnector
import statistics


class CollectionAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def top_collections_by_revenue(self):
        """
        Find top 10 film collections (>= 3 movies) by total revenue.
        Report: collection name, movie count, total revenue, median vote_average,
        earliest -> latest release date.
        """
        print("\n=== Query 4: Top Collections by Total Revenue ===\n")
        
        # MongoDB aggregation pipeline
        pipeline = [
            # Step 1: Filter movies that belong to a collection
            {
                "$match": {
                    "belongs_to_collection": {"$ne": None},
                    "belongs_to_collection.name": {"$exists": True, "$ne": None}
                }
            },
            
            # Step 2: Group by collection name
            {
                "$group": {
                    "_id": "$belongs_to_collection.name",
                    "movie_count": {"$sum": 1},
                    "total_revenue": {"$sum": "$revenue"},
                    "vote_averages": {"$push": "$vote_average"},
                    "release_dates": {"$push": "$release_date"},
                    "revenues": {"$push": "$revenue"}  # For filtering later
                }
            },
            
            # Step 3: Filter collections with >= 3 movies
            {"$match": {"movie_count": {"$gte": 3}}},
            
            # Step 4: Sort by total revenue descending
            {"$sort": {"total_revenue": -1}},
            
            # Step 5: Limit to top 10
            {"$limit": 10}
        ]
        
        results = list(self.db.movies.aggregate(pipeline))
        
        # Process results in Python for median and date range
        processed_results = []
        for collection in results:
            # Calculate median vote_average (filter out None values)
            vote_avgs = [v for v in collection['vote_averages'] if v is not None]
            median_vote = statistics.median(vote_avgs) if vote_avgs else None
            
            # Get earliest and latest release dates (filter out None)
            dates = [d for d in collection['release_dates'] if d is not None]
            dates.sort()
            earliest_date = dates[0] if dates else None
            latest_date = dates[-1] if dates else None
            
            processed_results.append({
                "collection_name": collection['_id'],
                "movie_count": collection['movie_count'],
                "total_revenue": collection['total_revenue'],
                "median_vote_average": round(median_vote, 2) if median_vote else None,
                "earliest_release": earliest_date,
                "latest_release": latest_date
            })
        
        # Print results
        print(f"{'Rank':<5} {'Collection':<40} {'Movies':<8} {'Total Revenue':<18} "
              f"{'Median Vote':<12} {'Release Span':<25}")
        print("-" * 115)
        
        for i, col in enumerate(processed_results, 1):
            date_span = f"{col['earliest_release']} → {col['latest_release']}" if col['earliest_release'] else "N/A"
            
            print(f"{i:<5} {col['collection_name']:<40} {col['movie_count']:<8} "
                  f"${col['total_revenue']:>15,.0f}  {col['median_vote_average']:<12} {date_span:<25}")
        
        return processed_results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = CollectionAnalysis()
        results = program.top_collections_by_revenue()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 4: Top Collections by Total Revenue ===

Rank  Collection                               Movies   Total Revenue      Median Vote  Release Span             
-------------------------------------------------------------------------------------------------------------------
1     Harry Potter Collection                  8        $  7,707,367,425  7.5          2001-11-16 → 2011-07-07  
2     Star Wars Collection                     8        $  7,434,494,790  7.45         1977-05-25 → 2016-12-14  
3     James Bond Collection                    26       $  7,106,970,239  6.3          1962-10-04 → 2015-10-26  
4     The Fast and the Furious Collection      8        $  5,125,098,793  6.65         2001-06-22 → 2017-04-12  
5     Pirates of the Caribbean Collection      5        $  4,521,576,826  6.9          2003-07-09 → 2017-05-23  
6     Transformers Collection                  5        $  4,366,101,244  6.1          2007-06-27 → 2017-06-2